# Industry-Style Agentic RAG with LangGraph

## Why this is industry-style

A simple RAG system searches only the private knowledge base.

An industry-style Agentic RAG system should do this:

```text
Question
  ↓
Search private knowledge base first
  ↓
Grade the private evidence
  ↓
If private evidence is good → answer from KB
  ↓
If private evidence is weak → search the web
  ↓
Grade web evidence
  ↓
Generate a grounded answer with source type
```

This design is useful because private documents may be incomplete or outdated.

## 1. Install Dependencies

## 2. Configure API Keys

In [31]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except EnvironmentError:
    raise

_GROQ_API_KEY = os.getenv("GROQ_API_KEY")
_GROQ_MODEL = os.getenv("GROQ_MODEL")
_TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
_PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
_GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
_LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
_HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
_EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME")
_PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME")
_PINECONE_INDEX_NAMESPACE = os.getenv("PINECONE_INDEX_NAMESPACE")
missing_api_key_list = []
if not _GROQ_API_KEY:
    missing_api_key_list.append("GROQ_API_KEY")
if not _GROQ_MODEL:
    missing_api_key_list.append("GROQ_MODEL")
if not _TAVILY_API_KEY:
    missing_api_key_list.append("TAVILY_API_KEY")
if not _PINECONE_API_KEY:
    missing_api_key_list.append("PINECONE_API_KEY")
if not _LANGSMITH_API_KEY:
    missing_api_key_list.append("LANGSMITH_API_KEY")
if not _GEMINI_API_KEY:
    missing_api_key_list.append("GEMINI_API_KEY")
if not _HUGGINGFACE_API_KEY:
    missing_api_key_list.append("HUGGINGFACE_API_KEY")
if not _EMBEDDING_MODEL_NAME:
    missing_api_key_list.append("EMBEDDING_MODEL_NAME")
if not _PINECONE_INDEX_NAME:
    missing_api_key_list.append("PINECONE_INDEX_NAME")
if not _PINECONE_INDEX_NAMESPACE:
    missing_api_key_list.append("PINECONE_INDEX_NAMESPACE")

if missing_api_key_list is None:
    print("missing_api_key_list: ", missing_api_key_list)
else:
    print("No missing_api_keys")

No missing_api_keys


In [9]:
print(_EMBEDDING_MODEL_NAME)

sentence-transformers/all-MiniLM-L6-v2


In [ ]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# making the pinecone object
pinecone_object= Pinecone(api_key=_PINECONE_API_KEY)

# # loading pinecone index
index_name = pinecone_object.Index(_PINECONE_INDEX_NAME)

# connecting the vector store using pinecone 
vector_store= PineconeVectorStore(
    index=index_name,
    embedding=_EMBEDDING_MODEL_NAME,
    namespace=_PINECONE_INDEX_NAMESPACE
)

# making retriver
retriever= vector_store.as_retriever(
    search_kwargs={"k":5,'namespace':_PINECONE_INDEX_NAMESPACE}
)

print("Existing Index: ",_PINECONE_INDEX_NAME)

Existing Index:  it-support-agentic-rag-indexname


## 3. Load Documents

create private knowledge base from a real web document:

`https://docs.langchain.com/oss/python/langgraph/agentic-rag`


The web fallback is separate. That means:

- First source: indexed LangGraph documentation page/private knowledge base
- Fallback source: live web search through Tavily

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
url= "https://docs.langchain.com/langsmith/python/managed-deep-agents-overview"
loader= WebBaseLoader(
    web_path=url,
)
raw_docs=loader.load()
print(raw_docs)
print("Length of data:",len(raw_docs))
print("source: ",raw_docs[0].metadata.get("source"))
print("\nPreview:")
print(raw_docs[0].page_content[:1500])

ImportError: cannot import name 'WebBaseLoader' from 'langchain_core.document_loaders' (f:\DsWithBuppy_Agentic AI Course： Build Al Agents & Automate Workflows\IT-Support-Agentic-RAG\IT-Support-Agentic-RAG\venv\Lib\site-packages\langchain_core\document_loaders\__init__.py)

## 4. Split the document into chunks

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter= RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True
)

chunks= splitter.split_documents(raw_docs)
print("No of chunks: ",len(chunks))
print("\nChunk Overview:\n")
print(chunks[0].page_content[:900])

No of chunks:  13

Chunk Overview:

Managed Deep Agents - Docs by LangChain


## 5.Create Free Local Embeddings

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model= HuggingFaceEmbeddings(
    model_name=_EMBEDDING_MODEL_NAME,
    encode_kwargs={"normalize_embeddings": True},
)
sample_vector= embedding_model.embed_query("What is AI")
print(sample_vector)
print("dimention of sample_vector: ", len(sample_vector))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1443.22it/s]


[-0.028200605884194374, -0.017675522714853287, 0.011596136726439, 0.01601574569940567, 0.045705925673246384, -0.016152145341038704, 0.06501340121030807, 0.031805988401174545, 0.029278932139277458, 0.05825493484735489, -0.042929694056510925, -0.003123966511338949, 0.009426412172615528, -0.04228344187140465, -0.044077642261981964, 0.07152608036994934, -0.024430735036730766, -0.04126305133104324, -0.07005303353071213, -0.0744963139295578, -0.013285133987665176, 0.00889817625284195, -0.05688982084393501, -0.07097277790307999, -0.03665563091635704, 0.09129757434129715, 0.028841977939009666, -0.07827780395746231, -0.010502156801521778, -0.01822674460709095, 0.01755453273653984, -0.06115095689892769, 0.08683191984891891, 0.015249386429786682, -0.040124934166669846, 0.02645130082964897, -0.04383980855345726, -0.04861941933631897, 0.0956171303987503, -0.02186714857816696, -0.014999916777014732, -0.040611714124679565, 0.01712803728878498, -0.0696665421128273, 0.10470209270715714, 0.1244667321443

In [13]:
def embedding_model_dimention(embedding_model):
    sample_vector= embedding_model.embed_query("What is AI?")
    dimention= len(sample_vector)
    return dimention


print("embedding_model_dimention: ",embedding_model_dimention(embedding_model))

embedding_model_dimention:  384


## 6. creating the pinecone vector database

The embedding model used in this notebook produces **384-dimensional vectors**, so the Pinecone index must use dimension `384`.

# steps:
# 1. create pinecone vector store or index using Pinecone class
# 2. send the documents to pinecone index using lagchain_pinecone class and making vector store
# 3. make the retriever

In [14]:
from pinecone import Pinecone,ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

# 1. create pinecone vector store or index using Pinecone class
pinecone_object= Pinecone(
    api_key=_PINECONE_API_KEY,
)
# creating the pinecone index if it doesn't exist
existing_indexes= [index_info["name"] for index_info in pinecone_object.list_indexes()]
print(existing_indexes)

if _PINECONE_INDEX_NAME not in existing_indexes:
    pinecone_object.create_index(
        name=_PINECONE_INDEX_NAME,
        dimension=embedding_model_dimention(embedding_model),
        metric='cosine',
        spec=ServerlessSpec(cloud='aws',region='us-east-1')
    )
# wait until the index is prepared in pinecone
while not pinecone_object.describe_index(name=_PINECONE_INDEX_NAME).status['ready']:
    time.wait(1)
print("pinecone index is ready: ",_PINECONE_INDEX_NAME)    

# 2. send the documents to pinecone index using lagchain_pinecone class and making vector store
vector_store= PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    index_name=_PINECONE_INDEX_NAME,
    namespace=_PINECONE_INDEX_NAMESPACE
)

# 3. make the retriever
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace":_PINECONE_INDEX_NAMESPACE
    }
)
print('Pinecone vector database and its retriever is ready.')

['it-support-agentic-rag-indexname']
pinecone index is ready:  it-support-agentic-rag-indexname
Pinecone vector database and its retriever is ready.


# 7.Loading the existing index

In [16]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# making the pinecone object
pinecone_object= Pinecone(api_key=_PINECONE_API_KEY)

# # loading pinecone index
index_name = pinecone_object.Index(_PINECONE_INDEX_NAME)

# connecting the vector store using pinecone 
vector_store= PineconeVectorStore(
    index=index_name,
    embedding=embedding_model,
    namespace=_PINECONE_INDEX_NAMESPACE
)

# making retriver
retriever= vector_store.as_retriever(
    search_kwargs={"k":5,'namespace':_PINECONE_INDEX_NAMESPACE}
)

print("Existing Index: ",_PINECONE_INDEX_NAME)

Existing Index:  it-support-agentic-rag-indexname


# 8. testing the private KB

In [29]:
query="what is langchain?"

kb_docs= retriever.invoke(query)

for i, doc in enumerate(kb_docs):
    print(f"\n----KB DOCS{i+1}---")
    print("source: ", doc.metadata.get('source'))
    print("content: ", doc.page_content[:500])



----KB DOCS1---
source:  https://docs.langchain.com/langsmith/python/managed-deep-agents-overview
content:  Managed Deep Agents - Docs by LangChain

----KB DOCS2---
source:  https://docs.langchain.com/langsmith/python/managed-deep-agents-overview
content:  Managed Deep Agents - Docs by LangChain

----KB DOCS3---
source:  https://docs.langchain.com/langsmith/python/managed-deep-agents-overview
content:  Managed Deep Agents - Docs by LangChain

----KB DOCS4---
source:  https://docs.langchain.com/langsmith/python/managed-deep-agents-overview
content:  Managed Deep Agents - Docs by LangChain

----KB DOCS5---
source:  https://docs.langchain.com/langsmith/python/managed-deep-agents-overview
content:  # Research

Use this skill when a task needs more than a direct answer.

1. Identify what information is missing.
2. Search LangChain docs when the question is about LangChain, LangGraph, or LangSmith.
3. Summarize findings before responding to the user.
tools/search.pyfrom langchain.tools impo

# 8. initializing groq LLM

In [47]:
from langchain_groq import ChatGroq
LLM= ChatGroq(
    model=_GROQ_MODEL,
    temperature=0
)
query= LLM.invoke("What is langchain in 10 words?")
print(query.content)

LangChain is a framework for building AI‑powered language applications efficiently.


# 9. Initiate Tavily websearch

Tavily will be used only when the private knowledge base is insufficient.

This is important because we do not want to send every private/company question to the public web.

In [38]:
from langchain_tavily import TavilySearch

web_search= TavilySearch(
    max_results= 5,
    topic= 'general',
    include_answer= True,
    # include_raw_content=True,
    # include_images=True
)

print("Tavily web_search tool is ready")

Tavily web_search tool is ready


In [42]:
web_search.invoke("recent AI news?")

{'query': 'recent AI news?',
 'follow_up_questions': None,
 'answer': "Recent AI news includes the expansion of UAE-based AI and data startup CNTXT AI into Silicon Valley, raising $60 million in June and developing Arabic-first AI models. Another notable development is the launch of doubleAI, an Nvidia-backed startup that claims to have reduced one customer's compute costs by 67%. Meanwhile, Google's AI faced a glitch refusing to answer who won the 2020 U.S. presidential election, and Google is aware of this issue and working on a fix. Additionally, Google launched the Gemini 4 Carbon AI model with a cybersecurity focus.",
 'images': [],
 'results': [{'url': 'https://guides.library.georgetown.edu/ai/news',
   'title': 'Guides: Artificial Intelligence (Generative) Resources: Latest News on AI',
   'content': "U.A.E.-Based AI and Data Startup Expands Into Silicon Valley; CNTXT AI, which raised $60 million in June, has built Arabic-first AI models and is training robots This link opens in

# 10. Defining structured Decisions and Langgraph state

We need reliable yes/no or route outputs from the LLM.

Structured outputs help avoid parsing messy natural language.

In [ ]:
from typing import List, Literal, TypedDict
from pydantic import BaseModel,Field
from langchain_core.documents import Document

class DecisionRoute(BaseModel):
    route: Literal["Kb",'direct']= Field(
        description="Using 'kb' when we need to answer using Agentic RAG; Use 'direct' for simple questions where 'kb' is not required."
    )

class GradingAnswer(BaseModel):
    grading: Literal['good','weak']= Field(
        description="grade the answer as 'good' if the evidence is enough to answer the query, otherwise return 'weak'."
    )

class AgentState(TypedDict):
    query: str
    source_used:str
    rewritten_query: str

    retrived_kb_docs: List[Document]
    web_results:str

    kb_grade:str
    web_search_grade:str
    answer: str   

    
    retry_count:int 

# Nodes

## 11. Node 1

The router decides:

- `kb`: use private knowledge base
- `direct`: answer without retrieval

In a real company assistant, this step may also route between many sources:
HR docs, finance docs, product docs, SQL database, web search, etc.

In [ ]:
router_llm= LLM.with_structured_output(DecisionRoute,method='json_mode')

def route_question(state:AgentState):
    query= state["query"]

    decision= router_llm.invoke(f'''
    you are a router for Agentic Rag assistant.

    route to 'kb' if the user asks about:
    - Agentic RAG
    - LangGraph Agentic RAG workflow
    - retrieval grading
    - query rewriting
    - RAG architecture
    - retriever tools
    - web fallback in RAG

    route to 'direct' only for greetings, thanks or very simple conversations.

    query:
    {query}

    Return response as a valid json:
    Example:
    {{'route':'kb'}} 
    ''')

    return {
        'query':query,
        'source_used': decision.route
    }

def route_after_routing(state: AgentState)->Literal['retriever_kb','direct_answer']:
    if state['source_used']=='kb':
        return 'retriever_kb'
    return 'direct_answer'

# 12. Node-2 : Retrive from the private KB